# RealPDE — is the default ±5% SPS band too narrow? (minimal v6 probe)

v3 returns **no bounds**, so the scorer falls back to `pred ± 0.05*|pred|`. v6 is the **same frozen CNO** plus a fixed band `pred ± bound_frac*|pred|` returned in `info["lower"/"upper"]` on **every** step (ingestion enforces all-or-none). This notebook sweeps `bound_frac`, scores each width with the **official** SPS formula, and packs the winner — answering whether the default is too narrow for this predictor.

In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR} ]; then echo "Pulling {REPO_DIR}..."; cd {REPO_DIR} && git pull; else echo "Cloning {REPO_URL}..."; git clone {REPO_URL} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}
import sys, torch
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}, device', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Config (the only cell you edit)

`WIDTHS` are half-width fractions: `0.05` reproduces the scorer default, the rest test wider bands.

In [ ]:
VARIANT = 'submission_v6'
WIDTHS = [0.05, 0.10, 0.20, 0.30, 0.50]
ALPHAS = [0.10, 0.30, 0.50, 0.70, 1.00]
N_FILES, N_FRAMES, SEED = 30, 200, 42

## 2. Resolve real data

Prefer an attached real `test_real/` split with the official `mean_std_real.pt`; otherwise stage `N_FILES` trajectories from `train_real/` via `scripts/stage_real30.py`.

In [ ]:
from pathlib import Path
REPO = Path(REPO_DIR)
DATA_ROOT = next((c for c in ['/kaggle/input/datasets/nthday/realpde', '/kaggle/input/realpde'] if Path(c).exists()), None)
print(f'DATA_ROOT -> {DATA_ROOT}')
DATA_DIR = None
for root in ([Path(DATA_ROOT)] if DATA_ROOT else []):
    for sub in ['test_real', 'test']:
        tdir = root / sub
        stats = root / 'mean_std_real.pt'
        if tdir.is_dir() and any(tdir.glob('*.h5')) and stats.exists():
            DATA_DIR = root
            break
if DATA_DIR is not None:
    print(f'[ok] USING REAL test split at {DATA_DIR}')
    data_dir_path = DATA_DIR

In [ ]:
# Stage from train_real only when no real test split is attached.
import subprocess, shutil
STAGE_DIR = Path('/kaggle/working/real30')
if DATA_DIR is None:
    train_real = None
    if DATA_ROOT is not None:
        base = Path(DATA_ROOT) / 'train_real'
        if sorted(base.glob('*.h5')):
            train_real = base
        elif sorted(base.rglob('*.h5')):
            from collections import Counter
            train_real = Counter(p.parent for p in base.rglob('*.h5')).most_common(1)[0][0]
    print(f'train_real -> {train_real}')
    assert train_real is not None, 'attach the realpde dataset (test/ or train_real/)'
    if STAGE_DIR.exists():
        shutil.rmtree(STAGE_DIR)
    r = subprocess.run([sys.executable, 'scripts/stage_real30.py', '--src', str(train_real),
                        '--dst', str(STAGE_DIR), '--n-files', str(N_FILES),
                        '--frames', str(N_FRAMES), '--seed', str(SEED)], cwd=REPO)
    print(f'[stage] exit code: {r.returncode}')
    data_dir_path = STAGE_DIR
print(f'\nFINAL DATA_DIR = {data_dir_path}')
assert (Path(data_dir_path) / 'test_real').is_dir()

## 3. Frozen CNO predictions (one streaming pass)

Stage the `sim_real` CNO checkpoint as v6's `model.pth`, then collect denormalized preds/targets. Predictions are width-independent, so one pass serves the whole sweep.

In [ ]:
# Stage the real CNO checkpoint so get_ttt_model loads it (not the TinyForecaster fallback).
import shutil
sub_dir = REPO / 'submissions' / VARIANT
CKPT = None
if not (sub_dir / 'model.pth').exists():
    roots = [Path(DATA_ROOT) if DATA_ROOT else None, Path('/kaggle/input/realpde'), Path('/kaggle/working/checkpoints')]
    for r in roots:
        if r is None or not r.exists():
            continue
        cands = sorted([p for p in r.rglob('*.pth') if 'cno' in p.name.lower()],
                       key=lambda p: (0 if 'sim_real' in p.name.lower() else 1, p.name))
        if cands:
            CKPT = cands[0]
            break
    assert CKPT is not None, 'no CNO checkpoint found in attached data'
    shutil.copy(CKPT, sub_dir / 'model.pth')
    print(f'staged model.pth <- {CKPT}')
else:
    print('model.pth already staged')
    CKPT = sub_dir / 'model.pth'
sys.path.insert(0, str(sub_dir))
import submission as V6
mod = V6.get_ttt_model(str(sub_dir), device)
print('model ready on', device)

In [ ]:
import h5py, numpy as np, torch, os
IN_STEP, OUT_STEP, INTERVAL, SUB_S = 20, 20, 20, 2
HORIZON = IN_STEP + OUT_STEP
stats = Path(data_dir_path) / 'mean_std_real.pt'
mi, mt, si, st = torch.load(stats, map_location=device, weights_only=False)
one = torch.ones_like
mean_in, mean_tgt = mi.float(), mt.float()
std_in = torch.where(si == 0, one(si), si).float()
std_tgt = torch.where(st == 0, one(st), st).float()

def preprocess(x, y):
    return (x - mean_in[..., :x.shape[-1]]) / std_in[..., :x.shape[-1]], \
           (y - mean_tgt[..., :y.shape[-1]]) / std_tgt[..., :y.shape[-1]]

def denorm(p):
    return p * std_tgt[..., :p.shape[-1]] + mean_tgt[..., :p.shape[-1]]

def build_stream(base):
    tdir = Path(base) / 'test_real'
    files = sorted(f for f in os.listdir(tdir) if f.endswith('.h5'))
    entries, frames_of = [], {}
    for name in files:
        with h5py.File(tdir / name, 'r') as f:
            frames_of[name] = f['u'].shape[0]
        for t in range(0, frames_of[name] - HORIZON + 1, INTERVAL):
            entries.append((name, t))
    entries.sort(key=lambda e: (e[0], e[1]))
    stream, prev = [], None
    for name, tid in entries:
        with h5py.File(tdir / name, 'r') as f:
            end = min(tid + HORIZON, frames_of[name])
            u = f['u'][tid:end, ::SUB_S, ::SUB_S]
            v = f['v'][tid:end, ::SUB_S, ::SUB_S]
        p = np.zeros_like(u)
        data = np.stack([u, v, p], axis=-1)
        if data.shape[0] < HORIZON:
            data = np.concatenate([data, np.repeat(data[-1:], HORIZON - data.shape[0], 0)], 0)
        stream.append({'input': data[:IN_STEP], 'target': data[IN_STEP:HORIZON], 'is_first': name != prev})
        prev = name
    return stream

stream = build_stream(data_dir_path)
print(f'stream: {len(stream)} steps')
mean_np = mean_tgt.squeeze().cpu().numpy().astype(np.float32)
std_np = std_tgt.squeeze().cpu().numpy().astype(np.float32)

def denorm_np(a):  # normalized-space array -> raw space (same affine map as denorm)
    return (a * std_np + mean_np).astype(np.float32)


In [ ]:
preds, tgts, predn = [], [], []
prev_pair = None
for s in stream:
    inp = torch.tensor(s['input'], dtype=torch.float32, device=device).unsqueeze(0)
    tgt = torch.tensor(s['target'], dtype=torch.float32, device=device).unsqueeze(0)
    if s['is_first']:
        mod.reset_ttt_state(); prev_pair = None
    in_n, tg_n = preprocess(inp, tgt)
    pred_n, _ = mod.ttt_step(in_n, prev_pair[1] if prev_pair else None)
    prev_pair = (in_n, tg_n)
    preds.append(denorm(torch.as_tensor(pred_n).detach()).squeeze(0).cpu().numpy().astype(np.float32))
    predn.append(torch.as_tensor(pred_n).detach().squeeze(0).cpu().numpy().astype(np.float32))
    tgts.append(tgt.squeeze(0).cpu().numpy().astype(np.float32))
pred_all = np.stack(preds, 0)[..., :2]
tgt_all = np.stack(tgts, 0)[..., :2]
predn_all = np.stack(predn, 0)
print('pred/target (u,v only):', pred_all.shape)

## 4. Width sweep with the official SPS formula

Bands are built in **normalized** space (`predn ± w*|predn|`, exactly what v6 returns) then denormalized with the same affine map the evaluator applies — so these numbers are what Codabench will score. `w=0.05` is the scorer default. Coverage is what widening buys; `mean_nil` (width / `SIGMA_GLOBAL`) is what it costs via `exp(-nil)`.

In [ ]:
import scoring as official
print(f"{'half-width':>12}{'total-width':>13}{'sps_score':>12}{'coverage':>10}{'mean_nil':>10}")
print('-' * 60)
best = None
for w in WIDTHS:
    lo = denorm_np(predn_all - w * np.abs(predn_all))[..., :2]
    hi = denorm_np(predn_all + w * np.abs(predn_all))[..., :2]
    sps, cov = official.aggregate_sps(pred_all, tgt_all, 2, lower=lo, upper=hi)
    nil = float(np.mean((hi - lo) / official.SIGMA_GLOBAL))
    score = official.score_sps(sps)
    print(f'{w:>12.3f}{2*w:>13.3f}{score:>12.2f}{cov:>10.4f}{nil:>10.3f}')
    if best is None or score > best[1]:
        best = (w, score)
print(f'\nbest half-width: {best[0]} (sps {best[1]:.2f}) — pack this one in §6')

## 5. Same widths *through* `ttt_step` (contract check)

§4 builds the same bands in numpy. Here each width goes through v6's `ttt_step` (`info["lower"/"upper"]`, normalized space, model device) and is denormalized exactly like the prediction — the same path the evaluator uses. Asserts bounds on **every** step with pred shape (all-or-none).

In [ ]:
print(f"{'half-width':>12}{'sps_score':>12}{'coverage':>10}{'mean_nil':>10}  steps-with-bounds")
print('-' * 70)
for w in WIDTHS:
    mod.bound_frac = w  # same knob policy.yaml sets; no reload needed
    lo_all, hi_all, n_ok = [], [], 0
    prev_pair = None
    for s in stream:
        inp = torch.tensor(s['input'], dtype=torch.float32, device=device).unsqueeze(0)
        tgt = torch.tensor(s['target'], dtype=torch.float32, device=device).unsqueeze(0)
        if s['is_first']:
            mod.reset_ttt_state(); prev_pair = None
        in_n, tg_n = preprocess(inp, tgt)
        pred_n, info = mod.ttt_step(in_n, prev_pair[1] if prev_pair else None)
        prev_pair = (in_n, tg_n)
        assert info.get('lower') is not None and info.get('upper') is not None, 'bounds missing on a step'
        assert tuple(torch.as_tensor(info['lower']).shape) == tuple(pred_n.shape), 'lower shape != pred shape'
        assert tuple(torch.as_tensor(info['upper']).shape) == tuple(pred_n.shape), 'upper shape != pred shape'
        n_ok += 1
        lo_all.append(denorm(torch.as_tensor(info['lower']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
        hi_all.append(denorm(torch.as_tensor(info['upper']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
    lo = np.stack(lo_all, 0); hi = np.stack(hi_all, 0)
    sps, cov = official.aggregate_sps(pred_all, tgt_all, 2, lower=lo, upper=hi)
    nil = float(np.mean((hi - lo) / official.SIGMA_GLOBAL))
    print(f'{w:>12.3f}{official.score_sps(sps):>12.2f}{cov:>10.4f}{nil:>10.3f}  {n_ok}/{len(stream)}')
print('\n§5 must match §4 row-for-row (same bands, evaluator path). If wider wins in both, the default is too narrow.')

## 5b. `submission_v4` adaptive intervals on the same stream

Same frozen CNO and same stream, but v4's online per-channel q90 band (`policy.yaml`: `coverage`, `history`) instead of v6's fixed fraction. Same evaluator path — one row, directly comparable to the §4/§5 tables.

In [ ]:
# Load v4 by file path (NOT plain `import submission` — that name is taken by v6 above).
import importlib.util, shutil
V4 = 'submission_v4'
v4sub = REPO / 'submissions' / V4
if not (v4sub / 'model.pth').exists():
    shutil.copy(sub_dir / 'model.pth', v4sub / 'model.pth')
    print(f'staged {V4}/model.pth from {VARIANT}')
if str(v4sub) not in sys.path:
    sys.path.insert(0, str(v4sub))
spec = importlib.util.spec_from_file_location('v4_submission', v4sub / 'submission.py')
V4mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(V4mod)
m4 = V4mod.get_ttt_model(str(v4sub), device)
print('v4 ready:', type(m4).__name__)

In [ ]:
lo4, hi4, n_ok = [], [], 0
prev_pair = None
for s in stream:
    inp = torch.tensor(s['input'], dtype=torch.float32, device=device).unsqueeze(0)
    tgt = torch.tensor(s['target'], dtype=torch.float32, device=device).unsqueeze(0)
    if s['is_first']:
        m4.reset_ttt_state(); prev_pair = None
    in_n, tg_n = preprocess(inp, tgt)
    pred_n, info = m4.ttt_step(in_n, prev_pair[1] if prev_pair else None)
    prev_pair = (in_n, tg_n)
    assert info.get('lower') is not None and info.get('upper') is not None, 'bounds missing on a step'
    n_ok += 1
    lo4.append(denorm(torch.as_tensor(info['lower']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
    hi4.append(denorm(torch.as_tensor(info['upper']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
lo4 = np.stack(lo4, 0); hi4 = np.stack(hi4, 0)
sps4, cov4 = official.aggregate_sps(pred_all, tgt_all, 2, lower=lo4, upper=hi4)
nil4 = float(np.mean((hi4 - lo4) / official.SIGMA_GLOBAL))
print(f'v4 adaptive: sps {official.score_sps(sps4):.2f}, coverage {cov4:.4f}, mean_nil {nil4:.3f} ({n_ok}/{len(stream)} steps)')
print('compare against the §4/§5 rows above: higher sps wins; check whether adaptive beats the best fixed width.')

## 5c. `submission_v5` (relative-q90) and `submission_v7` (EMA) on the same stream

v5: magnitude-scaled band from a fixed `history`-window table (one row, policy defaults). v7: v4's absolute band with EMA memory — sweep `ema_alpha` (`1.0` = memoryless, smaller = longer memory). Same stream, same evaluator path.

In [ ]:
# v5 by file path (same clash-avoidance as v4 above).
V5 = 'submission_v5'
v5sub = REPO / 'submissions' / V5
if not (v5sub / 'model.pth').exists():
    shutil.copy(sub_dir / 'model.pth', v5sub / 'model.pth')
    print(f'staged {V5}/model.pth from {VARIANT}')
if str(v5sub) not in sys.path:
    sys.path.insert(0, str(v5sub))
spec5 = importlib.util.spec_from_file_location('v5_submission', v5sub / 'submission.py')
V5mod = importlib.util.module_from_spec(spec5)
spec5.loader.exec_module(V5mod)
m5 = V5mod.get_ttt_model(str(v5sub), device)
lo5, hi5, n_ok = [], [], 0
prev_pair = None
for s in stream:
    inp = torch.tensor(s['input'], dtype=torch.float32, device=device).unsqueeze(0)
    tgt = torch.tensor(s['target'], dtype=torch.float32, device=device).unsqueeze(0)
    if s['is_first']:
        m5.reset_ttt_state(); prev_pair = None
    in_n, tg_n = preprocess(inp, tgt)
    pred_n, info = m5.ttt_step(in_n, prev_pair[1] if prev_pair else None)
    prev_pair = (in_n, tg_n)
    assert info.get('lower') is not None and info.get('upper') is not None, 'bounds missing on a step'
    n_ok += 1
    lo5.append(denorm(torch.as_tensor(info['lower']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
    hi5.append(denorm(torch.as_tensor(info['upper']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
lo5 = np.stack(lo5, 0); hi5 = np.stack(hi5, 0)
sps5, cov5 = official.aggregate_sps(pred_all, tgt_all, 2, lower=lo5, upper=hi5)
nil5 = float(np.mean((hi5 - lo5) / official.SIGMA_GLOBAL))
print(f'v5 relative-q90: sps {official.score_sps(sps5):.2f}, coverage {cov5:.4f}, mean_nil {nil5:.3f} ({n_ok}/{len(stream)} steps)')

In [ ]:
# v7 EMA sweep — alpha is a runtime attribute (reset_ttt_state clears the EMA, keeps alpha).
V7 = 'submission_v7'
v7sub = REPO / 'submissions' / V7
if not (v7sub / 'model.pth').exists():
    shutil.copy(sub_dir / 'model.pth', v7sub / 'model.pth')
    print(f'staged {V7}/model.pth from {VARIANT}')
if str(v7sub) not in sys.path:
    sys.path.insert(0, str(v7sub))
spec7 = importlib.util.spec_from_file_location('v7_submission', v7sub / 'submission.py')
V7mod = importlib.util.module_from_spec(spec7)
spec7.loader.exec_module(V7mod)
m7 = V7mod.get_ttt_model(str(v7sub), device)
print(f"{'ema_alpha':>12}{'sps_score':>12}{'coverage':>10}{'mean_nil':>10}  steps-with-bounds")
print('-' * 70)
best_a = None
for a in ALPHAS:
    m7.ema_alpha = a
    lo7, hi7, n_ok = [], [], 0
    prev_pair = None
    for s in stream:
        inp = torch.tensor(s['input'], dtype=torch.float32, device=device).unsqueeze(0)
        tgt = torch.tensor(s['target'], dtype=torch.float32, device=device).unsqueeze(0)
        if s['is_first']:
            m7.reset_ttt_state(); prev_pair = None
        in_n, tg_n = preprocess(inp, tgt)
        pred_n, info = m7.ttt_step(in_n, prev_pair[1] if prev_pair else None)
        prev_pair = (in_n, tg_n)
        assert info.get('lower') is not None and info.get('upper') is not None, 'bounds missing on a step'
        n_ok += 1
        lo7.append(denorm(torch.as_tensor(info['lower']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
        hi7.append(denorm(torch.as_tensor(info['upper']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
    lo7 = np.stack(lo7, 0); hi7 = np.stack(hi7, 0)
    sps7, cov7 = official.aggregate_sps(pred_all, tgt_all, 2, lower=lo7, upper=hi7)
    nil7 = float(np.mean((hi7 - lo7) / official.SIGMA_GLOBAL))
    score7 = official.score_sps(sps7)
    print(f'{a:>12.2f}{score7:>12.2f}{cov7:>10.4f}{nil7:>10.3f}  {n_ok}/{len(stream)}')
    if best_a is None or score7 > best_a[1]:
        best_a = (a, score7)
print(f'\nbest ema_alpha: {best_a[0]} (sps {best_a[1]:.2f}) — packed in §6')

## 5d. `submission_v8` (speed-conditioned q90) on the same stream

v4's table, but q90 is conditioned on predicted speed: residuals binned by `|prev_pred|` magnitude, per-channel q90 per bin, linear-interpolated at each current pixel's speed. No EMA. One row, same evaluator path.

In [ ]:
# v8 by file path (same clash-avoidance as v4/v5/v7 above).
V8 = 'submission_v8'
v8sub = REPO / 'submissions' / V8
if not (v8sub / 'model.pth').exists():
    shutil.copy(sub_dir / 'model.pth', v8sub / 'model.pth')
    print(f'staged {V8}/model.pth from {VARIANT}')
if str(v8sub) not in sys.path:
    sys.path.insert(0, str(v8sub))
spec8 = importlib.util.spec_from_file_location('v8_submission', v8sub / 'submission.py')
V8mod = importlib.util.module_from_spec(spec8)
spec8.loader.exec_module(V8mod)
m8 = V8mod.get_ttt_model(str(v8sub), device)
lo8, hi8, n_ok = [], [], 0
prev_pair = None
for s in stream:
    inp = torch.tensor(s['input'], dtype=torch.float32, device=device).unsqueeze(0)
    tgt = torch.tensor(s['target'], dtype=torch.float32, device=device).unsqueeze(0)
    if s['is_first']:
        m8.reset_ttt_state(); prev_pair = None
    in_n, tg_n = preprocess(inp, tgt)
    pred_n, info = m8.ttt_step(in_n, prev_pair[1] if prev_pair else None)
    prev_pair = (in_n, tg_n)
    assert info.get('lower') is not None and info.get('upper') is not None, 'bounds missing on a step'
    n_ok += 1
    lo8.append(denorm(torch.as_tensor(info['lower']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
    hi8.append(denorm(torch.as_tensor(info['upper']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
lo8 = np.stack(lo8, 0); hi8 = np.stack(hi8, 0)
sps8, cov8 = official.aggregate_sps(pred_all, tgt_all, 2, lower=lo8, upper=hi8)
nil8 = float(np.mean((hi8 - lo8) / official.SIGMA_GLOBAL))
print(f'v8 speed-q90: sps {official.score_sps(sps8):.2f}, coverage {cov8:.4f}, mean_nil {nil8:.3f} ({n_ok}/{len(stream)} steps)')
print('compare vs v4 (§5b), v5/v7 (§5c), best fixed (§4/§5): does conditioning beat the global band?')

## 6. `local_eval` sanity + pack the winner

`local_eval.py` runs the full harness (timing + all five subscores) against the staged `real30`/real test split. Then pack one zip per width — `--set bound_frac=` rewrites the policy inside the zip, no code edit.

In [ ]:
# Full-harness sanity on the policy default (bound_frac=0.05): mirrors the official streaming loop.
r = subprocess.run([sys.executable, 'local_eval.py', '--submission', f'submissions/{VARIANT}',
                    '--data', str(data_dir_path), '--device', device], cwd=REPO)
print(f'[{VARIANT} local_eval] exit code: {r.returncode}')

In [ ]:
# Pack one Codabench zip per swept width (submission.py at root, shared files injected).
for w in WIDTHS:
    tag = f'{w:.2f}'.replace('.', 'p')
    r = subprocess.run([sys.executable, 'scripts/make_submission_zip.py', VARIANT,
                        '--with-model', str(CKPT), '--set', f'bound_frac={w}',
                        '--out', f'dist/{VARIANT}_w{tag}.zip'], cwd=REPO)
    print(f'[pack w={w}] exit code: {r.returncode}')
# Pack v4 too (§5b contender, same checkpoint) so the Codabench pick is one upload away.
r = subprocess.run([sys.executable, 'scripts/make_submission_zip.py', V4,
                    '--with-model', str(CKPT), '--out', f'dist/{V4}_cno.zip'], cwd=REPO)
print(f'[pack {V4}] exit code: {r.returncode}')
# Pack one v7 zip per swept alpha (--set rewrites ema_alpha inside the zip).
for a in ALPHAS:
    tag = f'{a:.2f}'.replace('.', 'p')
    r = subprocess.run([sys.executable, 'scripts/make_submission_zip.py', V7,
                        '--with-model', str(CKPT), '--set', f'ema_alpha={a}',
                        '--out', f'dist/{V7}_a{tag}.zip'], cwd=REPO)
    print(f'[pack {V7} a={a}] exit code: {r.returncode}')
# Pack v8 (§5d contender, same checkpoint).
r = subprocess.run([sys.executable, 'scripts/make_submission_zip.py', V8,
                    '--with-model', str(CKPT), '--out', f'dist/{V8}_cno.zip'], cwd=REPO)
print(f'[pack {V8}] exit code: {r.returncode}')
!ls -la dist/ | grep -E 'v6|v4|v7|v8'

## 7. Spatial error maps — signed bias (over vs under)

`err = pred − target` in raw space (u, v). Positive bias (**red**) = model predicts **higher** velocity than observed there; negative (**blue**) = **underpredicts**. Maps average over windows × time; the binned table checks whether bias depends on flow speed itself.

In [ ]:
err = pred_all - tgt_all  # (N, T, H, W, 2), raw space
bias = err.mean(axis=(0, 1))  # (H, W, 2) signed bias map
rmse = np.sqrt((err ** 2).mean(axis=(0, 1)))  # (H, W, 2)
abst = np.abs(tgt_all).mean(axis=(0, 1))  # mean |target| per pixel
for i, ch in enumerate(['u', 'v']):
    b = bias[..., i]
    print(f'{ch}: mean bias {b.mean():+.5f} | mean|bias| {np.abs(b).mean():.5f} | '
          f'pixels>0 {(b > 0).mean():.3f} | rmse {rmse[..., i].mean():.5f} | mean|tgt| {abst[..., i].mean():.5f}')
print('sign convention: bias>0 means the model predicts HIGHER velocity than observed.')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
vmax = max(np.abs(bias[..., 0]).max(), np.abs(bias[..., 1]).max())
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
for i, ch in enumerate(['u', 'v']):
    im = ax[0][i].imshow(bias[..., i], cmap='seismic', vmin=-vmax, vmax=vmax, aspect='auto')
    ax[0][i].set_title(f'bias {ch} (pred-tgt), red=over, vmax={vmax:.4f}')
    fig.colorbar(im, ax=ax[0][i], fraction=0.046)
    im2 = ax[1][i].imshow(rmse[..., i], cmap='magma', aspect='auto')
    ax[1][i].set_title(f'RMSE {ch}')
    fig.colorbar(im2, ax=ax[1][i], fraction=0.046)
fig.tight_layout()
fig.savefig('/kaggle/working/bias_maps.png', dpi=100)
print('saved /kaggle/working/bias_maps.png')

In [ ]:
# Does bias drift with flow speed? Bin by target-speed deciles: mean error + q90|err| per bin.
spd = np.sqrt((tgt_all ** 2).sum(axis=-1))  # (N, T, H, W)
eu, ev, s = err[..., 0].ravel(), err[..., 1].ravel(), spd.ravel()
qs = np.quantile(s, np.linspace(0, 1, 11))
print(f"{'speed bin':>22}{'covU@qg':>10}{'covV@qg':>10}{'mean bias_u':>14}{'q90|err_u|':>12}{'mean bias_v':>14}{'q90|err_v|':>12}{'count':>10}")
qg_u = float(np.quantile(np.abs(eu), 0.90))  # CURRENT global interval half-width (u, raw units)
qg_v = float(np.quantile(np.abs(ev), 0.90))  # ... (v)
print(f'global q90 half-widths: u {qg_u:.5f}, v {qg_v:.5f} (overall coverage ~= 0.90 by construction)')
for lo, hi in zip(qs[:-1], qs[1:]):
    m = (s >= lo) & (s <= hi)
    au, av = np.abs(eu[m]), np.abs(ev[m])
    print(f'[{lo:>8.4f}, {hi:>8.4f}]'
          f'{(au <= qg_u).mean():>10.4f}{(av <= qg_v).mean():>10.4f}'
          f'{eu[m].mean():>14.5f}{np.quantile(au, 0.90):>12.5f}'
          f'{ev[m].mean():>14.5f}{np.quantile(av, 0.90):>12.5f}{m.sum():>10d}')
print('read: cov@qg flat at ~0.90 across bins = well calibrated; a slope (fast bins far below 0.90) '
      '= the global interval misallocates width across flow states -> condition the band on speed.')